# 03 — Modelos Baseline y Evaluación
**CRISP-DM: Modelado + Evaluación** · Etapa 1: *Baseline (≥3 algoritmos) + tabla comparativa + interpretabilidad*

Requiere `pip install -r requirements.txt` (scikit-learn, xgboost, shap).
Usa `data/processed/dataset_modelado.csv` generado en el notebook 02.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))
import pandas as pd
from prediccion_precios import features as ft, evaluation as ev, config
from prediccion_precios import models_baseline as mb, interpretability as it
pd.set_option("display.width",160); pd.set_option("display.max_columns",60)

In [2]:
data = pd.read_csv(config.DATASET_MODELADO, parse_dates=[config.COL_FECHA])
n_antes = len(data)
# dropna=True: descarta el NaN estructural de lags/medias móviles en las
# primeras semanas de cada producto (documentado y tratado en el notebook 02)
X, y = ft.construir_matriz_modelado(data)
X = X.astype(float)
n_descartadas = n_antes - len(X)
print(f"Filas descartadas por NaN estructural (lags/medias móviles iniciales): "
      f"{n_descartadas} ({n_descartadas / n_antes * 100:.2f}%)")
X_train, X_test, y_train, y_test = ev.split_temporal(X, y)
print("X", X.shape, "| train", len(X_train), "| test", len(X_test))

Filas descartadas por NaN estructural (lags/medias móviles iniciales): 40 (4.85%)
X (785, 34) | train 628 | test 157


## 1. Entrenar los 3 baselines (Regresión Lineal, Random Forest, XGBoost)

In [3]:
modelos = mb.entrenar_todos(X_train, y_train)
list(modelos.keys())

['regresion_lineal', 'random_forest', 'xgboost']

## 2. Tabla comparativa de métricas (MAE / RMSE / MAPE / R²)

In [4]:
resultados = {n: ev.calcular_metricas(y_test, m.predict(X_test)) for n, m in modelos.items()}
tabla = ev.tabla_comparativa(resultados)
print("Meta objetivo: MAPE <", config.META_MAPE_OBJETIVO*100, "%")
tabla

Meta objetivo: MAPE < 15.0 %


,MAE,RMSE,MAPE_%,R2
Modelo,,,,
regresion_lineal,1.4209,2.2990,5.7303,0.9941
random_forest,1.7593,2.7454,6.2069,0.9916
xgboost,1.6975,2.5806,7.1450,0.9926


## 3. Validación cruzada temporal (ventana expansiva)
Se pasa la *fábrica* del modelo (`mb.FABRICAS[nombre]`) para re-crearlo en cada fold.

In [5]:
for nombre, fabrica in mb.FABRICAS.items():
    cv = ev.validacion_cruzada_temporal(fabrica, X, y, n_splits=config.CV_SPLITS)
    print(nombre, "-> MAPE %:", cv["MAPE_%"], "| R2:", cv["R2"])

regresion_lineal -> MAPE %: {'media': 18.8262, 'std': 23.3825} | R2: {'media': 0.4536, 'std': 1.0756}


random_forest -> MAPE %: {'media': 8.2271, 'std': 3.6959} | R2: {'media': 0.955, 'std': 0.0567}


xgboost -> MAPE %: {'media': 9.4933, 'std': 4.1656} | R2: {'media': 0.9196, 'std': 0.1096}


## 4. Interpretabilidad — importancia de variables + SHAP

In [6]:
mejor = modelos["xgboost"]
display(it.importancia_variables(mejor, X.columns).head(15))
it.graficar_importancia(mejor, X.columns)            # -> reports/figures/importancia_variables.png
it.explicar_shap(mejor, X_test)                       # -> reports/figures/shap_summary.png

,feature,importancia
0,media_movil_4,0.368482
1,precio_lag1,0.164747
2,media_movil_12,0.143490
3,precio_lag4,0.115933
4,rendimiento_kg_ha,0.077863
5,precio_lag8,0.061020
6,precio_lag2,0.047418
7,media_movil_8,0.008902
8,area_ha,0.003247
9,produccion_t,0.002617


PermutationExplainer explainer:   1%|          | 1/157 [00:00<?, ?it/s]

PermutationExplainer explainer:   2%|▏         | 3/157 [00:26<00:30,  4.98it/s]

PermutationExplainer explainer:   3%|▎         | 4/157 [00:26<00:44,  3.48it/s]

PermutationExplainer explainer:   3%|▎         | 5/157 [00:27<00:52,  2.87it/s]

...

PermutationExplainer explainer: 100%|██████████| 157/157 [01:08<00:00,  3.97it/s]

PermutationExplainer explainer: 158it [01:09,  3.99it/s]                         

PermutationExplainer explainer: 158it [01:09,  2.27it/s]


C:\Users\Ponce\Documents\Maching_Learning_UES\prediccion_precios_agricolas\src\prediccion_precios\interpretability.py:77: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, X_muestra, show=False)


WindowsPath('C:/Users/Ponce/Documents/Maching_Learning_UES/prediccion_precios_agricolas/reports/figures/shap_summary.png')

## 5. Guardar los modelos entrenados

In [7]:
for nombre, modelo in modelos.items():
    mb.guardar_modelo(modelo, nombre)
print("Modelos guardados en", config.MODELS_DIR)

Modelos guardados en C:\Users\Ponce\Documents\Maching_Learning_UES\prediccion_precios_agricolas\models


### Conclusiones de Etapa 1
> Indicar el mejor baseline, su MAPE frente a la meta (<15%), las variables más
> influyentes según SHAP y los próximos pasos hacia la Etapa 2 (optimización de
> hiperparámetros, LSTM/redes neuronales y ensemble).